# Methodology Exploration

## Goal
This notebook loads **all Cochrane PDFs in this folder**, extracts full text, and then applies the criteria extraction approach to capture:

- study design criteria
- participant criteria
- intervention criteria
- outcome criteria

We then realised that we didn't need all of that for now, so I then scrape just for the selection criteria which is a condensed version of the entire criteria extraction + the objectives which provide a general idea of the goal for the reviews. 

This version does **not** use the old CSV loading step. It works directly from local PDFs.

In [ ]:
#Note to self Run this in before ->  pip install -q pypdf pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [11]:
from pathlib import Path
import re
import pandas as pd
from tqdm.auto import tqdm
from pypdf import PdfReader

In [ ]:
PROJECT_DIR = Path.cwd()
pdf_files = sorted(PROJECT_DIR.glob("CD*.pdf"))



Found 17 PDF files in: /Users/cyrilestier/Desktop/Capstone
First files: ['CD007651.pdf', 'CD007825.pdf', 'CD008552.pdf', 'CD009467.pdf', 'CD009604.pdf']


In [ ]:
def extract_text_from_pdf(pdf_path: Path) -> tuple[str, int]:
    reader = PdfReader(str(pdf_path))
    page_texts = []
    for page in reader.pages:
        page_texts.append(page.extract_text() or "")
    full_text = "\n".join(page_texts)
    return full_text, len(reader.pages)


records = []
for pdf_path in tqdm(pdf_files, desc="Reading PDFs"):
    full_text, page_count = extract_text_from_pdf(pdf_path)
    records.append(
        {
            "pdf_name": pdf_path.name,
            "review_id": pdf_path.stem,
            "page_count": page_count,
            "full_text": full_text,
            "full_text_len": len(full_text),
        }
    )

reviews_df = pd.DataFrame(records)
reviews_df[["review_id", "page_count", "full_text_len"]].head()

Reading PDFs:   0%|          | 0/17 [00:00<?, ?it/s]

reviews_df shape: (17, 5)


,review_id,page_count,full_text_len
0,CD007651,360,1156828
1,CD007825,138,439538
2,CD008552,283,824107
3,CD009467,58,230235
4,CD009604,91,364367


## Extraction Approach

1. Normalize OCR-heavy text.
2. Find candidate criteria blocks using section starts like `Criteria for considering studies for this review`.
3. Stop each candidate at nearby method boundaries (for example `Search methods...`, `Data collection and analysis`).
4. Choose the best candidate based on expected subheading coverage.
5. Split into structured sections for downstream prompting.

In [14]:
WS_RE = re.compile(r"\s+")

START_PATTERNS = [
    r"criteria\s+for\s+considering\s+studies\s+for\s+this\s+review",
    r"selection\s+criteria",
]

END_PATTERNS = [
    r"search\s+methods\s+for\s+identification\s+of\s+studies",
    r"data\s+collection\s+and\s+analysis",
    r"assessment\s+of\s+risk\s+of\s+bias",
]

SECTION_PATTERNS = {
    "study_design_criteria": [r"types\s+of\s+studies", r"study\s+designs?"],
    "participants_criteria": [r"types\s+of\s+participants?"],
    "interventions_criteria": [r"types\s+of\s+interventions?"],
    "outcomes_criteria": [r"types\s+of\s+outcome\s+measures?", r"types\s+of\s+outcomes?"],
}

ABSTRACT_SECTION_PATTERNS = [
    r"background",
    r"objectives?",
    r"search\s+methods?",
    r"selection\s+criteria",
    r"data\s+collection\s+and\s+analysis",
    r"main\s+results?",
    r"authors'?\s+conclusions?",
]


def normalize_ocr_text(text: str) -> str:
    text = str(text)
    text = text.replace("ﬁ", "fi").replace("ﬂ", "fl").replace("ﬀ", "ff").replace("ﬃ", "ffi")
    text = text.replace("•", " - ")
    text = re.sub(r"---\s*Page\s*\d+\s*---", " ", text, flags=re.I)
    text = WS_RE.sub(" ", text)
    return text.strip()


def find_pattern_positions(text_lower: str, patterns: list[str]) -> list[int]:
    positions = []
    for pattern in patterns:
        positions.extend(match.start() for match in re.finditer(pattern, text_lower))
    return sorted(set(positions))


def extract_best_criteria_block(raw_text: str, max_fallback_chars: int = 12000) -> str:
    text = normalize_ocr_text(raw_text)
    text_lower = text.lower()

    starts = find_pattern_positions(text_lower, START_PATTERNS)
    if not starts:
        return ""

    candidates = []
    for start in starts:
        ends = []
        for pattern in END_PATTERNS:
            ends.extend(start + 1 + m.start() for m in re.finditer(pattern, text_lower[start + 1 :]))

        ends = [e for e in ends if e - start > 150]
        end = min(ends) if ends else min(len(text), start + max_fallback_chars)

        block = text[start:end].strip()
        block_lower = block.lower()

        heading_hits = 0
        for patterns in SECTION_PATTERNS.values():
            heading_hits += int(any(re.search(p, block_lower) for p in patterns))

        candidates.append((heading_hits, len(block), block))

    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return candidates[0][2]


def extract_heading_block(
    raw_text: str,
    heading_patterns: list[str],
    max_fallback_chars: int = 2500,
    search_window_chars: int = 200000,
) -> str:
    text = normalize_ocr_text(raw_text)
    text_lower = text.lower()

    starts = find_pattern_positions(text_lower, heading_patterns)
    if not starts:
        return ""

    starts_in_window = [s for s in starts if s < search_window_chars]
    starts = starts_in_window if starts_in_window else starts

    candidates = []
    for start in starts:
        ends = []
        for pattern in ABSTRACT_SECTION_PATTERNS:
            ends.extend(start + 1 + m.start() for m in re.finditer(pattern, text_lower[start + 1 :]))

        ends = [e for e in ends if e - start > 40]
        next_heading_end = min(ends) if ends else len(text)
        end = min(next_heading_end, start + max_fallback_chars)

        block = text[start:end].strip()
        word_count = len(block.split())
        dot_ratio = block.count(".") / max(1, len(block))
        toc_like = bool(re.search(r"\.{4,}", block[:250]))

        candidates.append((start, block, word_count, dot_ratio, toc_like))

    clean_candidates = [
        c for c in candidates
        if c[2] >= 12 and c[3] <= 0.08 and not c[4]
    ]

    chosen = min(clean_candidates, key=lambda x: x[0]) if clean_candidates else min(candidates, key=lambda x: x[0])
    return chosen[1]


def extract_objectives_text(raw_text: str) -> str:
    text = extract_heading_block(raw_text, [r"\bobjectives\b"])
    if not text:
        text = extract_heading_block(raw_text, [r"\bobjective\b"])
    return text


def extract_selection_criteria_text(raw_text: str) -> str:
    return extract_heading_block(raw_text, [r"selection\s+criteria"])


def first_match_pos(text_lower: str, patterns: list[str]):
    matches = []
    for pattern in patterns:
        match = re.search(pattern, text_lower)
        if match:
            matches.append(match.start())
    return min(matches) if matches else None


def split_criteria_sections(criteria_block: str) -> dict:
    criteria_lower = criteria_block.lower()
    located = []

    for section_name, patterns in SECTION_PATTERNS.items():
        pos = first_match_pos(criteria_lower, patterns)
        if pos is not None:
            located.append((pos, section_name))

    located.sort()
    out = {name: "" for name in SECTION_PATTERNS}

    for i, (start_pos, section_name) in enumerate(located):
        end_pos = located[i + 1][0] if i + 1 < len(located) else len(criteria_block)
        out[section_name] = criteria_block[start_pos:end_pos].strip()

    return out


def compact(text: str, max_chars: int = 200) -> str:
    text = str(text).strip()
    return text[:max_chars] + ("..." if len(text) > max_chars else "")



In [15]:
analysis_df = reviews_df.copy()

analysis_df["objectives_text"] = analysis_df["full_text"].map(extract_objectives_text)
analysis_df["selection_criteria_text"] = analysis_df["full_text"].map(extract_selection_criteria_text)

analysis_df["criteria_block"] = analysis_df["full_text"].map(extract_best_criteria_block)
analysis_df["criteria_block_len"] = analysis_df["criteria_block"].str.len()

section_df = analysis_df["criteria_block"].apply(split_criteria_sections).apply(pd.Series)
analysis_df = pd.concat([analysis_df, section_df], axis=1)

for col in [
    "objectives_text",
    "selection_criteria_text",
    "study_design_criteria",
    "participants_criteria",
    "interventions_criteria",
    "outcomes_criteria",
]:
    analysis_df[f"{col}_len"] = analysis_df[col].str.len()

print("Coverage across all reviews:")
for col in [
    "objectives_text",
    "selection_criteria_text",
    "criteria_block",
    "study_design_criteria",
    "participants_criteria",
    "interventions_criteria",
    "outcomes_criteria",
]:
    non_empty = (analysis_df[col].str.len() > 0).sum()
    print(f"- {col}: {non_empty}/{len(analysis_df)}")

analysis_df[
    [
        "review_id",
        "objectives_text_len",
        "selection_criteria_text_len",
        "criteria_block_len",
        "study_design_criteria_len",
        "participants_criteria_len",
        "interventions_criteria_len",
        "outcomes_criteria_len",
    ]
].sort_values("criteria_block_len", ascending=False)


Coverage across all reviews:
- objectives_text: 17/17
- selection_criteria_text: 15/17
- criteria_block: 17/17
- study_design_criteria: 17/17
- participants_criteria: 16/17
- interventions_criteria: 16/17
- outcomes_criteria: 16/17


,review_id,objectives_text_len,selection_criteria_text_len,criteria_block_len,study_design_criteria_len,participants_criteria_len,interventions_criteria_len,outcomes_criteria_len
12,CD012333,303,1326,11318,5209,909,1514,3634
7,CD010067,283,0,9774,3434,1579,1860,2849
10,CD010919,185,1262,8629,2448,546,1716,3867
13,CD012415,215,1848,8239,2458,829,2334,2566
0,CD007651,264,509,7927,1923,652,2710,2590
11,CD011779,684,943,7508,669,343,789,5655
8,CD010321,269,363,7459,1287,639,3270,2211
4,CD009604,221,628,7075,1351,540,2415,2717
16,CD013789,346,479,6788,1214,1060,2713,1749
3,CD009467,154,903,6739,1999,519,680,3489


The only issue here is the CD010734 review. Should check further into it.

In [16]:
preview_cols = [
    "review_id",
    "objectives_text",
    "selection_criteria_text",
    "study_design_criteria",
    "participants_criteria",
    "interventions_criteria",
    "outcomes_criteria",
]

preview_df = analysis_df[preview_cols].copy()
for col in preview_cols[1:]:
    preview_df[col] = preview_df[col].map(lambda x: compact(x, 180))

preview_df


,review_id,objectives_text,selection_criteria_text,study_design_criteria,participants_criteria,interventions_criteria,outcomes_criteria
0,CD007651,Objectives The purpose of this review update i...,Selection criteria Eligible interventions were...,Types of studies In accordance with the last u...,Types of participants Studies that included sc...,Types of interventions Interventions We includ...,"Types of outcome measures To be included, stud..."
1,CD007825,Objectives To evaluate the effects of interage...,Selection criteria Randomized controlled trial...,Types of studies Included studies were randomi...,Types of participants All population types and...,Types of interventions Any interventions of in...,Types of outcome measures The primary outcomes...
2,CD008552,Objectives To assess the benefits and harms of...,Selection criteria We included randomised cont...,Types of studies Eligible trials were randomis...,Types of participants Participants could inclu...,Types of interventions We considered any educa...,Types of outcome measures We included trials t...
3,CD009467,Objectives We aimed to assess the effects of a...,Selection criteria We included any randomized ...,Types of studies All stages of this review fol...,Types of participants We included studies that...,Types of interventions This review included st...,Types of outcome measures Based on our study p...
4,CD009604,Objectives To determine the effects and safety...,Selection criteria We included randomised cont...,Types of studies Fortification of condiments a...,Types of participants Participants included th...,Types of interventions We included interventio...,Types of outcome measures We included studies ...
5,CD009820,Objectives To assess the effects of WtW interv...,Selection criteria Randomised controlled trial...,Types of studies Due to the difficulties inher...,Types of participants Lone parents and their d...,Types of interventions We included welfare-to-...,Types of outcome measures The primary outcomes...
6,CD009905,Objectives To assess effects of community coal...,Selection criteria Cluster-randomized controll...,Types of studies We included cluster-randomize...,Types of participants Community-level coalitio...,Types of interventions Interventions include l...,Types of outcome measures We have included stu...
7,CD010067,objectives are as follows: The main objective ...,,Types of studies Slum upgrading programmes hav...,Types of participants Populations living in ur...,Types of interventions This review will examin...,Types of outcome measures The review will exam...
8,CD010321,Objectives The objective of this review was to...,Selection criteria We included randomised cont...,Types of studies We included all randomised co...,Types of participants The target population wa...,Types of interventions We defined elder abuse ...,Types of outcome measures The following are th...
9,CD010734,objectives are as follows: To assess the effec...,,Types of studies Food fortification is an inte...,Types of participants All populations (includi...,Types of interventions Interventions in this r...,Types of outcome measures We will include stud...


In [ ]:
TARGET_REVIEW_ID = "CD010919"  # change this value to inspect another PDF

row = analysis_df.loc[analysis_df["review_id"].str.lower() == TARGET_REVIEW_ID.lower()]
if row.empty:
    print(f"{TARGET_REVIEW_ID} not found. Available review IDs:")
    print(", ".join(analysis_df["review_id"].tolist()))
else:
    r = row.iloc[0]
    print(f"Review ID: {r['review_id']}")
    print(f"File: {r['pdf_name']}")
    print(f"Pages: {r['page_count']}")
    print(f"Full text length: {r['full_text_len']:,} chars")
    print(f"Objectives length: {r['objectives_text_len']:,} chars")
    print(f"Selection criteria length: {r['selection_criteria_text_len']:,} chars")
    print(f"Criteria block length: {r['criteria_block_len']:,} chars")

    print("\nOBJECTIVES")
    print(r["objectives_text"] if str(r["objectives_text"]).strip() else "[EMPTY]")

    print("\nSELECTION CRITERIA")
    print(r["selection_criteria_text"] if str(r["selection_criteria_text"]).strip() else "[EMPTY]")

    print("\nFULL CRITERIA BLOCK")
    print(r["criteria_block"])

    section_cols = [
        ("STUDY DESIGN", "study_design_criteria"),
        ("PARTICIPANTS", "participants_criteria"),
        ("INTERVENTIONS", "interventions_criteria"),
        ("OUTCOMES", "outcomes_criteria"),
    ]

#Asked AI on how to print it out the cleanest way: 
    for label, col in section_cols:
        print("\n" + "=" * 100)
        print(label)
        print("=" * 100)
        print(r[col] if str(r[col]).strip() else "[EMPTY]")


Review ID: CD010919
File: CD010919.pdf
Pages: 247
Full text length: 742,251 chars
Objectives length: 185 chars
Selection criteria length: 1,262 chars
Criteria block length: 8,629 chars

OBJECTIVES
Objectives To assess the effectiveness of interventions to reduce ambient particulate matter air pollution in reducing pollutant concentrations and improving associated health outcomes.

SELECTION CRITERIA
Selection criteria Eligible for inclusion were randomized and cluster randomized controlled trials, as well as several non-randomized study designs, including controlled interrupted time-series studies (cITS-EPOC), interrupted time-series studies adhering to EPOC standards (ITS-EPOC), interrupted time-series studies not adhering to EPOC standards (ITS), controlled before-a/f_ter studies adhering to EPOC standards (CBA- EPOC), and controlled before-a/f_ter studies not adhering to EPOC standards (CBA); these were classified as main studies. Additionally, we included uncontrolled before-a/f_te

In [19]:
output_path = PROJECT_DIR / "review_criteria_extraction.csv"
analysis_df.to_csv(output_path, index=False)
print(f"Saved full extraction: {output_path}")

phase2_context = (analysis_df[["review_id", "objectives_text", "selection_criteria_text"]]
                  .drop_duplicates(subset=["review_id"]))
phase2_path = PROJECT_DIR / "review_phase2_context.csv"
phase2_context.to_csv(phase2_path, index=False)
print(f"Saved Phase 2 context: {phase2_path}")

phase2_context.head(17)


Saved full extraction: /Users/cyrilestier/Desktop/Capstone/review_criteria_extraction.csv
Saved Phase 2 context: /Users/cyrilestier/Desktop/Capstone/review_phase2_context.csv


,review_id,objectives_text,selection_criteria_text
0,CD007651,Objectives The purpose of this review update i...,Selection criteria Eligible interventions were...
1,CD007825,Objectives To evaluate the effects of interage...,Selection criteria Randomized controlled trial...
2,CD008552,Objectives To assess the benefits and harms of...,Selection criteria We included randomised cont...
3,CD009467,Objectives We aimed to assess the effects of a...,Selection criteria We included any randomized ...
4,CD009604,Objectives To determine the effects and safety...,Selection criteria We included randomised cont...
5,CD009820,Objectives To assess the effects of WtW interv...,Selection criteria Randomised controlled trial...
6,CD009905,Objectives To assess effects of community coal...,Selection criteria Cluster-randomized controll...
7,CD010067,objectives are as follows: The main objective ...,
8,CD010321,Objectives The objective of this review was to...,Selection criteria We included randomised cont...
9,CD010734,objectives are as follows: To assess the effec...,


## Notes

- `analysis_df` contains one row per PDF review.
- `full_text` is the full extracted PDF text.
- `criteria_block` is the extracted methodology section.
- Structured criteria columns are ready for your inclusion/exclusion prompt pipeline.

If needed, you can now reuse `analysis_df` directly in the next notebook stage.